# STAC V2 — try the conversion pipeline in Colab

This notebook lets you **run STAC V2 yourself** (the ANN-to-SNN conversion framework) with no local setup.

**What to expect — read this first.** After the v4.0.0 fixes:

- ✅ **Conversion with spiking OFF is faithful** — the converted model reproduces the original transformer to floating-point precision.
- ❌ **Genuine spiking on a *frozen* model does not work yet** — turn spiking ON without retraining and the model collapses to near-constant output. That is the honest, documented result, and this notebook lets you see it directly.
- ⚠️ **Energy is worse, not better, at the current ~5% spike coverage** (projected ~7.6× worse than the dense model on this tiny scale).
- 🔬 **Training recovers quality** — §8 runs the full end-to-end retraining (convert → train → re-evaluate) that recovered perplexity ~10.5× in the study, and §9 loads the result and generates text.

So this is a research/diagnostic tool, not a working energy-efficient chatbot. Run the cells top to bottom.

> **Tip:** For the fine-tuning section, switch to a GPU runtime: **Runtime → Change runtime type → T4 GPU**. Everything else runs fine on CPU.

## 1. Check the runtime (optional)

Shows whether you have a GPU. Not required for conversion/energy cells; recommended for fine-tuning.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU. For the fine-tuning section, use Runtime > Change runtime type > GPU.')

## 2. Get the code and install dependencies

Clones the repo and installs the pinned dependencies. STAC caps `transformers < 4.48` and `numpy < 2.0`, so this may **downgrade** the versions Colab ships with.

**If pip prints a message about restarting**, do **Runtime → Restart session**, then run this cell again and continue. (You do not need to re-clone.)

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/iLevyTate/stac'
BRANCH = 'claude/codebase-updates-v2-status-p19lcw'

if not os.path.isdir('stac'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, 'stac'], check=True)

%cd /content/stac
!pip install -q -r requirements.txt
print('\nDependencies installed. If pip asked you to restart the runtime, do it now, then re-run this cell.')

In [ ]:
# Make the repo importable and confirm the versions are in the supported range.
import sys
sys.path.insert(0, '/content/stac')
import transformers, numpy, torch
print('transformers:', transformers.__version__, '(must be < 4.48)')
print('numpy       :', numpy.__version__, '(must be < 2.0)')
print('torch       :', torch.__version__)

## 3. Run the test suite (proves the fixes are in place)

This builds tiny offline model fixtures and runs the pytest suite — including the liveness tests that assert spikes actually exist and predictions vary by position (the checks whose absence hid the original bugs). Takes a couple of minutes on CPU.

In [ ]:
!python scripts/make_test_models.py --out local/test-models
!python -m pytest tests/ -q

## 4. Convert DistilGPT-2 — spiking OFF (should be faithful)

Converts a real model and compares the converted logits against the original. With spiking off, the difference should be at the level of floating-point noise (top-1 agreement ~100%).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from smollm2_converter import simplified_conversion

name = 'distilgpt2'
tok = AutoTokenizer.from_pretrained(name)
orig = AutoModelForCausalLM.from_pretrained(name).eval()

# Convert with spiking OFF (skip_gelu_replacement=True preserves the original computation).
conv_off = simplified_conversion(AutoModelForCausalLM.from_pretrained(name).eval(),
                                 timesteps=8, skip_gelu_replacement=True, real_spiking=False)

ids = tok('The capital of France is', return_tensors='pt').input_ids
with torch.no_grad():
    lo = orig(ids).logits[:, -1]
    lc = conv_off(ids).logits[:, -1]

max_diff = (lo - lc).abs().max().item()
top1_agree = (lo.argmax(-1) == lc.argmax(-1)).float().mean().item()
print(f'max abs logit difference: {max_diff:.2e}')
print(f'top-1 agreement         : {top1_agree*100:.1f}%')
print('Original  next-token:', tok.decode(lo.argmax(-1)))
print('Converted next-token:', tok.decode(lc.argmax(-1)))

## 5. Convert with spiking ON — see the collapse and the energy projection

Now route Q/K/V through the LIF neurons (`real_spiking=True`). Two things to notice:

1. **`measure_spikes(...).summary()`** reports real spikes and an operation-count energy projection. The `(…x)` at the end is ANN/SNN energy — **below 1.0 means the SNN is projected *worse*** than the dense model.
2. Outputs now **change with the timestep count** (T=1 vs T=8), which is the evidence the spiking loop is genuinely doing something — and also why a frozen model drifts / collapses without retraining.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from smollm2_converter import simplified_conversion
from spike_metrics import measure_spikes

m = AutoModelForCausalLM.from_pretrained('distilgpt2').eval()
spk = simplified_conversion(m, timesteps=8, skip_gelu_replacement=True, real_spiking=True)

probe = torch.randint(0, 200, (1, 32))
report = measure_spikes(spk, probe, use_cache=False)
print(report.summary())
print(f'\nenergy ratio (ANN/SNN) = {report.energy_ratio:.2f}x  ->',
      'SNN cheaper' if report.energy_ratio > 1 else 'SNN MORE EXPENSIVE than the dense model')

In [ ]:
# Evidence the timestep loop is not a no-op: logits differ between T=1 and T=8 under spiking.
import torch
from transformers import AutoModelForCausalLM
from smollm2_converter import simplified_conversion

ids = torch.randint(0, 200, (1, 16))
diffs = {}
for T in (1, 8):
    m = AutoModelForCausalLM.from_pretrained('distilgpt2').eval()
    conv = simplified_conversion(m, timesteps=T, skip_gelu_replacement=True, real_spiking=True)
    with torch.no_grad():
        diffs[T] = conv(ids).logits[:, -1]

print('max |logits(T=8) - logits(T=1)| =', (diffs[8] - diffs[1]).abs().max().item())
print('(A large value here = spiking changes the computation; float noise would be ~1e-7.)')

## 6. Why the energy is unfavourable — coverage, not sparsity

The closed-form analysis shows the timestep budget is set by **how much of the model is spike-driven (coverage)**, not by the spike rate. The current design only spikes ~5% of the math. The scaling sweep shows coverage improves on *larger* models (the `lm_head` shrinks relative to the body), which is why an advantage is reachable in principle on SmolLM2-1.7B but not on tiny models.

In [ ]:
!python scripts/energy_analysis.py --scaling

In [ ]:
!python scripts/energy_analysis.py --arch smollm2-1.7b --seq_len 2048

## 7. (Optional) SmolLM2-135M — the RoPE fix

The single biggest V2 bug was that rotary position embeddings (RoPE) were dropped, which wrecked Llama-family models like SmolLM2. This downloads the real 135M model and checks that conversion with spiking **off** reproduces it. (Downloads ~500MB.)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from smollm2_converter import simplified_conversion

name = 'HuggingFaceTB/SmolLM2-135M'
tok = AutoTokenizer.from_pretrained(name)
orig = AutoModelForCausalLM.from_pretrained(name).eval()
conv = simplified_conversion(AutoModelForCausalLM.from_pretrained(name).eval(),
                             timesteps=8, skip_gelu_replacement=True, real_spiking=False)

ids = tok('Neuromorphic computing is', return_tensors='pt').input_ids
with torch.no_grad():
    lo = orig(ids).logits[:, -1]
    lc = conv(ids).logits[:, -1]
print('max abs logit difference:', (lo - lc).abs().max().item())
print('top-1 agreement:', (lo.argmax(-1) == lc.argmax(-1)).float().mean().item())
print('(Near-zero difference here is the RoPE + RMSNorm fix working.)')

## 8. End-to-end retraining — the recovery path (GPU recommended)

**This is the full end-to-end run**, and the reason a Colab is worth having: the project's own audit could not finish it for lack of GPU hardware. `finetune_spiking.py` does the whole chain in one shot:

1. **Convert** the model and extend spike coverage across MLP / attention / lm_head.
2. **Calibrate** the spike thresholds.
3. **Train** through the T-timestep spiking forward with backprop-through-time, distilling against the original model as a teacher (`--distill`).
4. **Evaluate** perplexity before (`step 0`) and after (`final`) — that before/after gap *is* the end-to-end test.
5. **Save** the retrained spiking model (`--save`) so §9 can load it and generate text.

In the study, ~300 CPU steps recovered eval perplexity ~10.5× (6,161 → 580) where every frozen-weight remedy moved it *not at all*. It was still ~11× above the dense baseline at this tiny scale — so this is a **direction, not a finished result**. Push `--steps` to 1000–3000 on a GPU to go further.

> The trainer reads WikiText-2 via the `datasets` package, which isn't in the base requirements — the next cell installs it.

In [ ]:
!pip install -q 'datasets>=2.14.0'

In [ ]:
# Real end-to-end retrain. On a T4 GPU ~300 steps is a few minutes; on CPU it is slow,
# so --max_seconds stops it cleanly (still saving a checkpoint) before Colab's idle limit.
# For a faster CPU probe use '--timesteps 2 --seq_len 32 --steps 50 --components mlp'.
!python scripts/finetune_spiking.py --model distilgpt2 --distill --timesteps 8 \
    --steps 300 --seq_len 128 --eval_every 100 --max_seconds 1800 --save trained_snn.pt

## 9. (Optional) Load the retrained model and generate text

Rebuilds the spiking model exactly as the trainer did, loads the weights you just saved, and greedily generates a few tokens using the T-averaged spiking logits. This is a **qualitative smoke test** — at distilgpt2 scale with a short run the text will be rough, but it lets you see the retrained spiking model actually produce output rather than collapse to a single repeated token.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from smollm2_converter import (simplified_conversion, TemporalSpikeProcessor,
                               calibrate_spike_attention)
from spike_coverage import apply_spike_coverage, calibrate_thresholds
from spikingjelly.activation_based import functional

name, T = 'distilgpt2', 8
tok = AutoTokenizer.from_pretrained(name)

# Reconstruct the same architecture the trainer built, then load the trained weights.
conv = simplified_conversion(AutoModelForCausalLM.from_pretrained(name).eval(),
                             T, skip_gelu_replacement=True, real_spiking=False)
inner = conv.snn_model if isinstance(conv, TemporalSpikeProcessor) else conv
apply_spike_coverage(inner, ['mlp', 'attn_qkv_proj', 'attn_out_proj', 'lm_head'], signed=True)
calib = [tok('The capital of France is', return_tensors='pt').input_ids]
calibrate_thresholds(conv, calib)
calibrate_spike_attention(conv, calib)

missing, unexpected = conv.load_state_dict(torch.load('trained_snn.pt', map_location='cpu'),
                                           strict=False)
print(f'loaded weights (missing={len(missing)}, unexpected={len(unexpected)})')
conv.eval()

def spiking_next_logits(ids):
    functional.reset_net(inner)
    acc = None
    for _ in range(T):
        out = inner(ids, use_cache=False)
        lg = out.logits if hasattr(out, 'logits') else out[0]
        acc = lg if acc is None else acc + lg
    return (acc / T)[:, -1]

ids = tok('The capital of France is', return_tensors='pt').input_ids
for _ in range(20):
    with torch.no_grad():
        nxt = spiking_next_logits(ids).argmax(-1, keepdim=True)
    ids = torch.cat([ids, nxt], dim=1)
print(repr(tok.decode(ids[0])))

---
### Recap

| You ran | What it shows |
|---|---|
| §3 tests | The fixes are in and guarded by regression tests |
| §4 spiking OFF | Conversion is faithful to the original model |
| §5 spiking ON | Real spikes happen, but energy projects **worse** and frozen output drifts |
| §6 energy | The advantage needs high spike *coverage*, reachable only on large models |
| §7 SmolLM2 | The RoPE/RMSNorm fix restored Llama-family conversion |
| §8 retrain | **End-to-end**: convert → train → the before/after perplexity gap that frozen conversion never closes |
| §9 generate | The retrained spiking model producing text instead of collapsing |

**Bottom line on "end to end":** yes — §8 converts, retrains, and re-evaluates in one run, and §9 loads the result and generates. What it demonstrates is *perplexity recovery* at tiny scale, not a deployable assistant. A conclusive result needs a bigger model, more steps, and a GPU — exactly what this notebook lets you try.

Full write-up: `docs/findings-summary.md` in the repo.